# Development of a Convolutional Neural Network (CNN) for Dog–Cat Image Classification Using Three Convolutional Layers

**Author:** Computer Vision & Machine Learning Student Project  
**Target Environment:** Google Colab / Local Python 3.11+ / TensorFlow 2.16+ (Keras 3)  
**Architecture:** Exactly THREE Convolutional (`Conv2D`) Layers with Binary Sigmoid Output  
**Dataset:** Microsoft Cats vs Dogs Dataset (~25,000 genuine photographic images)  

---

### Project Objective & Workflow
This notebook provides a complete, self-contained, end-to-end reproducible workflow for binary classification between cats (`0`) and dogs (`1`).

Key Pipeline Highlights:
1. **Corrupt Image Sanitization**: Specifically addresses corrupted EXIF headers, 0-byte files, and unreadable markers (such as `PetImages/Cat/666.jpg` and `PetImages/Dog/11702.jpg`).
2. **Elimination of Unknown Rank Errors**: Uses static tensor shape enforcement (`set_shape([128, 128, 3])`) to prevent Keras 3 runtime rank errors.
3. **Zero Data Leakage**: Enforces strict stratified splitting (70% Train, 15% Validation, 15% Test) prior to training.

## 1. Environment Setup & Dependency Verification
Install required dependencies if executing in a bare Google Colab instance and verify hardware accelerator (GPU/CPU).

In [ ]:
# Optional installation for Colab environments
# !pip install --quiet tensorflow pillow matplotlib scikit-learn pyyaml

import os
import sys
import random
import shutil
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Fix random seeds for reproducible runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[GPU Detected] Running with GPU acceleration: {gpus}")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
else:
    print("[CPU Mode] No GPU detected; executing on CPU.")

## 2. Dataset Download & Archive Verification
Download the genuine Microsoft Research Cats vs Dogs archive (`kagglecatsanddogs_5340.zip`) directly from the official Microsoft server.

In [ ]:
DATASET_URL = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
RAW_DIR = Path("data/raw")
ZIP_FILE = RAW_DIR / "kagglecatsanddogs_5340.zip"
RAW_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_FILE.is_file() or ZIP_FILE.stat().st_size == 0:
    print(f"Downloading dataset from {DATASET_URL}...")
    req = urllib.request.Request(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(ZIP_FILE, "wb") as out:
        shutil.copyfileobj(resp, out)
    print(f"Download complete: {ZIP_FILE.stat().st_size / (1024*1024):.2f} MB")
else:
    print(f"Dataset archive already exists locally: {ZIP_FILE}")

PET_IMAGES_DIR = RAW_DIR / "PetImages"
if not (PET_IMAGES_DIR / "Cat").is_dir() or not (PET_IMAGES_DIR / "Dog").is_dir():
    print("Extracting dataset archive...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as archive:
        archive.extractall(RAW_DIR)
    print("Extraction complete.")
else:
    print(f"Extracted PetImages directory verified at: {PET_IMAGES_DIR}")

## 3. Data Integrity Sanitization & Corrupt Image Handling
The raw Microsoft dataset is notorious for containing corrupt JPEG headers, zero-byte truncated files (e.g., `Dog/11702.jpg`), non-JPEG formats, and invalid EXIF metadata.
Here we validate every file using Pillow's `verify()`, re-open it to strip malformed EXIF data, convert to 3-channel RGB, and save clean copies to `data/processed`.

In [ ]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["Cat", "Dog"]
CLASS_INDICES = {"Cat": 0, "Dog": 1}

sanitization_report = {"Cat": {"valid": 0, "corrupt": 0}, "Dog": {"valid": 0, "corrupt": 0}}
valid_records = []

for class_name in CLASSES:
    raw_class_dir = PET_IMAGES_DIR / class_name
    proc_class_dir = PROCESSED_DIR / class_name
    proc_class_dir.mkdir(parents=True, exist_ok=True)
    
    raw_files = sorted([f for f in raw_class_dir.iterdir() if f.is_file() and not f.name.startswith('.')])
    print(f"Scanning {len(raw_files)} raw files for class '{class_name}'...")
    
    for f in raw_files:
        if f.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            sanitization_report[class_name]["corrupt"] += 1
            continue
        
        target_file = proc_class_dir / f"{class_name}_{f.stem}.jpg"
        
        # Check if already processed
        if target_file.is_file() and target_file.stat().st_size > 0:
            sanitization_report[class_name]["valid"] += 1
            valid_records.append((str(target_file), CLASS_INDICES[class_name], class_name))
            continue
            
        if f.stat().st_size == 0:
            sanitization_report[class_name]["corrupt"] += 1
            continue
            
        try:
            # Phase 1: verify file integrity
            with Image.open(f) as img:
                img.verify()
            # Phase 2: re-open and convert to clean RGB
            with Image.open(f) as img:
                rgb_img = img.convert("RGB")
                rgb_img.save(target_file, format="JPEG", quality=95)
            sanitization_report[class_name]["valid"] += 1
            valid_records.append((str(target_file), CLASS_INDICES[class_name], class_name))
        except Exception:
            sanitization_report[class_name]["corrupt"] += 1

print("\n--- Sanitization Summary ---")
for c in CLASSES:
    print(f"{c}: {sanitization_report[c]['valid']} Valid, {sanitization_report[c]['corrupt']} Rejected/Corrupt")
print(f"Total Clean Valid Images: {len(valid_records)}")

## 4. Stratified Data Partitioning (Zero-Leakage Splitting)
We split the validated image paths into 70% Training, 15% Validation, and 15% Testing with a fixed random seed (`42`).  
We assert programmatically that no image path overlaps between splits.

In [ ]:
paths = [r[0] for r in valid_records]
labels = [r[1] for r in valid_records]

# Step 1: Split 70% Train vs 30% Temp
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED
)

# Step 2: Split 30% Temp equally into 15% Val and 15% Test
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED
)

# Strict assertion to guarantee zero data leakage
assert len(set(train_paths).intersection(set(val_paths))) == 0, "Data leakage between Train and Val!"
assert len(set(train_paths).intersection(set(test_paths))) == 0, "Data leakage between Train and Test!"
assert len(set(val_paths).intersection(set(test_paths))) == 0, "Data leakage between Val and Test!"

print("Splits successfully created with zero leakage:")
print(f"  Training:   {len(train_paths)} samples ({len(train_paths)/len(paths)*100:.1f}%)")
print(f"  Validation: {len(val_paths)} samples ({len(val_paths)/len(paths)*100:.1f}%)")
print(f"  Testing:    {len(test_paths)} samples ({len(test_paths)/len(paths)*100:.1f}%)")

## 5. High-Performance tf.data Input Pipeline (Preventing Unknown Rank)
To resolve the Keras error *"Cannot take the length of shape with unknown rank"*, we explicitly call `set_shape` on both the decoded image tensor `[128, 128, 3]` and scalar label tensor `[]`.  
Pixel values are normalized to `[0.0, 1.0]` by dividing by `255.0`.

In [ ]:
IMG_HEIGHT, IMG_WIDTH = 128, 128
BATCH_SIZE = 32

def parse_image(file_path, label, is_training=False):
    raw = tf.io.read_file(file_path)
    img = tf.io.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = tf.cast(img, tf.float32) / 255.0
    
    if is_training:
        img = tf.image.random_flip_left_right(img)
        
    # Enforce static shape definition to eliminate unknown rank errors
    img.set_shape([IMG_HEIGHT, IMG_WIDTH, 3])
    label = tf.cast(label, tf.float32)
    label.set_shape([])
    return img, label

def create_dataset(paths, labels, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((tf.constant(paths), tf.constant(labels)))
    if is_training:
        ds = ds.shuffle(buffer_size=min(len(paths), 1024), seed=SEED)
    ds = ds.map(lambda p, l: parse_image(p, l, is_training=is_training), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_paths, train_labels, is_training=True)
val_ds = create_dataset(val_paths, val_labels, is_training=False)
test_ds = create_dataset(test_paths, test_labels, is_training=False)

print(f"Dataset element specifications: {train_ds.element_spec}")

## 6. Model Definition: Exactly THREE Conv2D Layers
The model architecture is built strictly with THREE Convolutional (`Conv2D`) layers:
1. **Conv2D Block 1**: 32 filters (3×3), ReLU, same padding + MaxPooling2D (2×2) -> `(64, 64, 32)`
2. **Conv2D Block 2**: 64 filters (3×3), ReLU, same padding + MaxPooling2D (2×2) -> `(32, 32, 64)`
3. **Conv2D Block 3**: 128 filters (3×3), ReLU, same padding + MaxPooling2D (2×2) -> `(16, 16, 128)`
4. **Dense Classifier**: Flatten -> Dense(128, ReLU) -> Dropout(0.5) -> Dense(1, Sigmoid)

In [ ]:
inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="input_image")

# Layer 1: Low-level edges, color transitions
x = layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv2d_1")(inputs)
x = layers.MaxPooling2D((2, 2), name="maxpool_1")(x)

# Layer 2: Mid-level textures, corners, fur patterns
x = layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2d_2")(x)
x = layers.MaxPooling2D((2, 2), name="maxpool_2")(x)

# Layer 3: High-level semantic structures (ears, snout, eyes)
x = layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv2d_3")(x)
x = layers.MaxPooling2D((2, 2), name="maxpool_3")(x)

# Classification Head
x = layers.Flatten(name="flatten")(x)
x = layers.Dense(128, activation="relu", name="dense_1")(x)
x = layers.Dropout(0.5, name="dropout")(x)
outputs = layers.Dense(1, activation="sigmoid", name="output_dense")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="DogCat_3Layer_CNN")

# Architecture verification
conv_layers_count = sum(1 for l in model.layers if isinstance(l, layers.Conv2D))
assert conv_layers_count == 3, f"Architecture Error: Expected 3 Conv2D layers, found {conv_layers_count}"

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

## 7. Model Training & Validation
Train the network with `ModelCheckpoint`, `EarlyStopping`, and `ReduceLROnPlateau`.

In [ ]:
EPOCHS = 20
CHECKPOINT_PATH = "artifacts/best_model.keras"
Path("artifacts").mkdir(parents=True, exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor="val_loss", save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 8. Training & Validation Visualizations
Plot and analyze training dynamics to check for overfitting/underfitting.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='#1976D2', lw=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='#E64A19', lw=2, linestyle='--')
plt.title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='#1976D2', lw=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', color='#E64A19', lw=2, linestyle='--')
plt.title('Training & Validation Loss', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()

## 9. Rigorous Evaluation on the Isolated Test Set
Compute test loss, test accuracy, precision, recall, F1-score, and display the confusion matrix and ROC curve.

In [ ]:
# Load best restored weights
best_model = keras.models.load_model(CHECKPOINT_PATH)

print("Evaluating on isolated test set...")
eval_res = best_model.evaluate(test_ds, verbose=1)
test_loss, test_acc = eval_res[0], eval_res[1]

preds = best_model.predict(test_ds, verbose=1).flatten()
y_true = np.array(test_labels)
y_pred = (preds >= 0.5).astype(np.int32)

print("\n" + "="*50)
print(f"Test Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Precision:     {precision_score(y_true, y_pred):.4f}")
print(f"Recall:        {recall_score(y_true, y_pred):.4f}")
print(f"F1-Score:      {f1_score(y_true, y_pred):.4f}")
print(f"ROC AUC:       {roc_auc_score(y_true, preds):.4f}")
print("="*50)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Cat", "Dog"]))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap=plt.cm.Blues)
plt.colorbar(im)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Cat", "Dog"])
ax.set_yticklabels(["Cat", "Dog"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontweight="bold")
plt.title("Test Set Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## 10. Single-Image Custom Prediction Function
Test the deployed model on any uploaded or local image path with proper validation.

In [ ]:
def predict_custom_image(image_path, model):
    img = Image.open(image_path).convert("RGB")
    img_resized = img.resize((IMG_WIDTH, IMG_HEIGHT), Image.Resampling.BILINEAR)
    tensor = np.array(img_resized, dtype=np.float32) / 255.0
    tensor = np.expand_dims(tensor, axis=0)
    
    prob_dog = float(model.predict(tensor, verbose=0)[0][0])
    predicted_class = "Dog" if prob_dog >= 0.5 else "Cat"
    confidence = prob_dog if prob_dog >= 0.5 else (1.0 - prob_dog)
    
    plt.imshow(img)
    plt.title(f"Predicted: {predicted_class} ({confidence*100:.2f}% confidence)", fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    return predicted_class, confidence

# Example prediction on a test sample
sample_image = test_paths[0]
predict_custom_image(sample_image, best_model)